# 02d — Microbiology features

Calculate culture timing and ESBL status around admission. Duplicate spells are reduced before assigning episode-level features.

In [25]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import pandas as pd
from pathlib import Path
from icare_risk.utils.paths import get_path, get_data_root_from_schema
from icare_risk.utils.preprocessing import keep_first_spell

# Define schema configuration to use.
SCHEMA_CFG = 'configs/schema_1000p.yaml'
PHENOTYPES_CFG = 'configs/phenotypes.yaml'

# Paths
DATA_DIR = get_data_root_from_schema(schema_path=get_path(SCHEMA_CFG)) 
DATA_PROC_DIR = DATA_DIR / 'processed'
DATA_PROC_DIR.mkdir(parents=True, exist_ok=True)

# Debug
print(f"Using data from: {DATA_DIR}")

Using data from: /app/data/synthetic/10000p


In [28]:
# --- OPTION A: Load using pandas ---
from icare_risk.clinphen.io.loaders import load_from_pandas

# Load dataframes
dfs = load_from_pandas(
    schema_path=SCHEMA_CFG,
    domains_to_load=['episodes', 'microbiology'],
    verbose=1
)

✅ Schema validation passed! All files exist and all physical columns match successfully.
Loaded... episodes (2.46MB) shape (10000, 11)
Loaded... microbiology (4.89MB) shape (9947, 10)


In [29]:
# --- OPTION B: Load using DuckDB ---
from icare_risk.clinphen.io.loaders import load_from_duckdb

# Configure
domains = ['episodes', 'microbiology']

# Load
dfs = load_from_duckdb(
    schema_path=SCHEMA_CFG,
    phenotype_path=PHENOTYPES_CFG,
    domains_to_load=domains,
    custom_codes={'microbiology': None}, # Bypass filtering 
    skip_mapping=True, 
    case_sensitive=False, verbose=1
)

# Assign
df_episodes = keep_first_spell(dfs['episodes'])
df_micro = dfs['microbiology']

✅ Schema validation passed! All files exist and all physical columns match successfully.
Skipping 'definitions': missing module or function definition.
Loaded... episodes (2.46MB) shape (10000, 11)
Loaded... microbiology (4.89MB) shape (9947, 10) | Filtered codes: None
episodes: 10,000 rows -> 10,000 unique spells


In [30]:
# Libraries
from icare_risk.clinphen.utils import (
    compute_time_to_closest_culture,
    assign_esbl_to_episodes,
)
from icare_risk.utils.paths import get_path

# Compute time to closest culture
df_micro_timing = compute_time_to_closest_culture(
    df_episodes=df_episodes,
    microbiology_df=df_micro,
    pre_admission_days=14,
    post_discharge_days=7,
    round_decimals=0,
)

# Define lookup paths
LOOKUP_3GCR_PATH = get_path('configs/lookups/3GCR.csv')
LOOKUP_3GCS_PATH = get_path('configs/lookups/3GCS.csv')

# Assign ESBL labels to episodes
df_microbiology = assign_esbl_to_episodes(
    df_episodes=df_micro_timing,
    df_microbiology=df_micro,
    path_3gcr=str(LOOKUP_3GCR_PATH),
    path_3gcs=str(LOOKUP_3GCS_PATH),
    pre_admission_days=14,
    post_discharge_days=7,
)
df_microbiology = keep_first_spell(df_microbiology, "microbiology features")
display(df_microbiology.head())
display(df_microbiology.esbl_status.value_counts())


Computing test-level ESBL labels on the fly...
microbiology features: 10,000 rows -> 10,000 unique spells


,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,INDEX_OF_MULTIPLE_DEPRIVATION_DECILE,closest_culture_date,time_to_culture_days,time_to_culture_hours,esbl_status
0,10001,7896381,9619750,5881100,2020-09-29,13:57:00,43,2020-10-07,140,oral surgery,7,NaN,NaN,NaN,unknown
1,10002,1805970,5717880,3457211,2020-08-03,19:47:00,49,2020-08-15,320,cardiology,unknown,NaN,NaN,NaN,unknown
2,10003,6698303,3675573,4694133,2022-11-02,16:32:00,32,2022-11-11,320,cardiology,9,NaN,NaN,NaN,unknown
3,10004,6804249,7908451,5949276,2023-01-05,02:07:00,59,2023-01-10,300,general medicine,9,NaN,NaN,NaN,unknown
4,10005,2301493,9166223,5218519,2020-11-13,19:34:00,51,2020-12-04,320,cardiology,unknown,NaN,NaN,NaN,unknown


esbl_status
unknown     9980
non-ESBL      10
ESBL          10
Name: count, dtype: int64

In [31]:
filename = "02e_derived_features_microbiology.parquet"
output_path = DATA_PROC_DIR / filename
df_microbiology.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")

Saved: /app/data/synthetic/10000p/processed/02e_derived_features_microbiology.parquet
